# Comparing Student Loan Repayment Plans with PROC LOAN

## Executive Summary

A financial-aid office evaluates how a graduating cohort should repay a representative $27,500 balance by comparing four repayment structures — a federal fixed-rate standard plan, a private refinance with an origination fee, a capped variable-rate (ARM) loan, and an employer-sponsored buydown — using `PROC LOAN`. Over a 120-month term the four offers line up cleanly on monthly payment and total interest at their quoted starting rates: the federal standard plan costs the most interest (**$10,021.22** at 6.53%, payment **$312.68**), the private refinance trims the rate but adds a $412.50 fee (**$9,120.20** at 5.99%, payment **$305.17**), and the lower-quoted ARM (**$7,099.76** at 4.75%, payment **$288.33**) and employer buydown (**$6,700.67** at 4.50%, payment **$285.01**) carry the smallest interest bills. A `COMPARE` block then reports each plan's cumulative interest, principal, and outstanding balance at 36, 60, and 120 months, showing how far each loan has amortized at the points a borrower is most likely to refinance or pay off.

## Data Sources

| Dataset | Rows | Description | Key Variables |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Synthetic graduating-cohort loan profiles generated inline with `call streaminit(20260531)` and `rand('uniform')`. Used to motivate realistic loan terms for the comparison. | `student_id` (1001-1040), `balance` (principal at graduation; this draw spans $11,800-$47,750, mean $30,771), `apr` (4.10%-9.10% annual nominal rate, mean 6.50%), `term` (120 or 180 months, mean 144), `origination_pct` (1.0%-2.0% fee, mean 1.50%) |

The representative borrower analyzed with `PROC LOAN` (amount $27,500, 120-month term, July 2026 start) sits near the lower-middle of this cohort's balance and rate distribution; no external or network data is used. The cohort exists to motivate plausible loan terms — the head-to-head comparison is run on the single representative loan.

# Comparing Student Loan Repayment Plans with PROC LOAN

When students graduate, a financial-aid office must help them choose among competing repayment offers. `PROC LOAN` (SAS/ETS) is purpose-built for exactly this: it amortizes fixed-rate, adjustable-rate (ARM), and buydown loans and then compares them side by side on payment, total interest, and amortization progress.

In this notebook we:

1. Generate a synthetic graduating cohort to establish realistic loan terms.
2. Summarize the cohort with `PROC MEANS`.
3. Build four repayment plans for a representative $27,500 balance and compare them with `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Re-run the recommended fixed plan on its own to confirm its stand-alone economics.

Everything runs locally on inline synthetic data — no external files or network access.

## Step 1 — Generate a synthetic graduating cohort

We simulate 40 borrowers. Each draws a principal balance at graduation, an APR tied loosely to credit quality, a standard repayment term (10 or 15 years), and an origination-fee percentage. The seed makes the data reproducible.

In [1]:
data borrowers;
   call streaminit(20260531);
   length plan $ 28;
   do student_id = 1001 to 1040;
      /* Principal balance at graduation: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Annual nominal APR by credit tier: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standard repayment term: 120 or 180 months */
      if rand('uniform') < 0.6 then term = 120;
      else term = 180;
      /* Origination fee as a percent of principal: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      output;
   end;
run;

NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


## Step 2 — Profile the cohort

Before modeling individual loans, we look at the distribution of balances, rates, and terms. This tells us what a *representative* borrower looks like — the basis for the head-to-head comparison that follows.

In [2]:
proc means data=borrowers n mean min max maxdec=2;
   var balance apr term origination_pct;
run;

                                                  The MEANS Procedure

 Variable               N           Mean     Minimum     Maximum
 ---------------------------------------------------------------
 BALANCE               40       30771.25    11800.00    47750.00
 APR                   40           6.50        4.10        9.10
 TERM                  40         144.00      120.00      180.00
 ORIGINATION_PCT       40           1.50        1.00        2.00
 ---------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Compare four repayment plans for a representative borrower

Using a representative $27,500 balance over a 120-month (10-year) term starting July 2026, we line up four realistic offers:

- **Federal Direct Unsubsidized (Standard)** — a plain fixed-rate loan at 6.53%.
- **Private Refinance (with fee)** — a lower 5.99% fixed rate, but with a $412.50 initialization cost (`INIT=`).
- **Private Variable ARM** — a 4.75% adjustable loan carrying a 1%-per-adjustment / 5% lifetime `CAPS=`, a 9.75% `MAXRATE=`, annual `ADJUSTFREQ=`, and the `WORSTCASE` keyword.
- **Employer 2-1 Buydown** — a subsidized 4.50% start that, on the quoted schedule, steps up via `BDRATES=` to 5.50% in year 2 and 6.50% thereafter.

The `COMPARE` statement asks for the cross-loan view at 36, 60, and 120 months, with a 22% `TAXRATE=` and a 4% minimum attractive rate of return (`MARR=`); `OUTSUM=` and `OUTCOMP=` capture the per-loan summary and the comparison rows. The listing below reports, for each plan and each checkpoint, the **cumulative interest paid, cumulative principal retired, and the outstanding balance** — a clear amortization-progress view across the candidates.

> **Note on rate adjustments.** Jenner's `PROC LOAN` amortizes every plan at its quoted **starting** rate, so the ARM's `CAPS`/`MAXRATE`/`WORSTCASE` settings and the buydown's `BDRATES` steps are echoed in the listing as the loan's terms but are **not** applied to the payment math — the ARM and buydown figures below reflect their 4.75% and 4.50% start rates held flat for the full term. Treat those two totals as best-case (start-rate) costs, not worst-case ceilings.

In [3]:
proc loan start=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Federal Direct Unsubsidized (Standard)';

   fixed rate=5.99 init=412.50
         label='Private Refinance (with fee)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='Private Variable ARM (worst case)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Employer 2-1 Buydown then 6.5%';

   compare at=(36 60 120) all taxrate=22 marr=4 outcomp=plan_cmp;
run;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federal Direct Unsubsidized (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Private Refinance (with fee)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Private Variable ARM (worst case)
    Loan Type:                   Adjustable Rate
    Amount (Principal):      

NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Step 4 — Re-run the recommended fixed plan on its own

For the borrower who values payment certainty, the federal fixed-rate standard plan is the safe default. We re-run it in isolation to confirm its headline economics: the same **$312.68** monthly payment, **$37,521.22** total paid, and **$10,021.22** total interest seen in the four-way comparison, now presented as a stand-alone loan summary.

In [4]:
proc loan start=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Federal Direct Unsubsidized (Standard)';
run;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federal Direct Unsubsidized (Standard)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Summary of Loan Comparisons
    Label                                Amount     Rate %   Months        Payment        Balance
    ------------------------------ ------------ ---------- -------- -------------- --------------
    Federal Direct Unsubsidized (S     27500.00     6.5300      120         312.68           0.00



NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpreting the results

All four plans fully amortize the same $27,500 principal over 120 months (every plan's outstanding balance reaches $0.00 at month 120 in the COMPARE table), so the decision turns on **monthly payment** and **total interest at the quoted rate**:

| Plan | Rate | Payment | Total interest | Notes |
|------|------|---------|----------------|-------|
| Federal Direct Standard | 6.53% | $312.68 | $10,021.22 | No fee; strongest borrower protections |
| Private Refinance | 5.99% | $305.17 | $9,120.20 | $412.50 origination fee |
| Private Variable ARM | 4.75% (start) | $288.33 | $7,099.76 | Rate can rise; figure is the start-rate cost |
| Employer 2-1 Buydown | 4.50% (start) | $285.01 | $6,700.67 | Depends on continued employment |

- The **federal standard** plan is the most expensive on interest ($10,021.22) but offers a fixed, predictable $312.68 payment with no fee.
- The **private refinance** shaves the rate to 5.99% ($9,120.20 interest, $901 less than the federal plan) but charges a $412.50 origination fee, so its net advantage over the federal plan is roughly $488 of interest minus fee — meaningful only if the loan runs long enough not to be refinanced away.
- The **ARM** and **buydown** show the lowest interest here ($7,099.76 and $6,700.67) purely because their **starting** rates are well below the fixed offers. As noted in Step 3, Jenner holds those start rates flat, so these are best-case figures: a real ARM that adjusts up — or a buydown that loses its employer subsidy — would land higher, and the payment is not guaranteed.

The `COMPARE` table sharpens this by showing how fast each plan amortizes. At 36 months the federal plan has paid $4,792.27 of interest and retired $6,464.10 of principal (balance $21,035.90), while the buydown has paid only $3,263.97 of interest and retired $6,996.24 of principal (balance $20,503.76) — the lower-rate plans both cost less interest and chip away at principal faster over the first three years. For a risk-averse graduate, the federal standard plan's rate certainty often justifies its higher interest; for a borrower confident in stable, lasting employment, the buydown's subsidized start delivers the largest savings among the options that actually fix their rate.